# Auditing a feasible schedule against a certified bound

- **Level:** 2, Fundamentals
- **Prerequisites:** 01-problem-and-published-data
- **Reading time:** about 12 minutes

## Objectives

Use PhaseFlow’s manifest reader to compare feasible CPIT rows with the certified bound, and test basic order and gap invariants.

## Setup

Environment pins come from `notebooks/requirements.txt`. The product implementation remains in `data-pipeline/`; this notebook imports its manifest helper.

```bash
pip install oreblocks==0.6.3 numpy==2.5.1 scipy==1.18.1 matplotlib==3.11.2
```

In [1]:
# 02 bound and feasible plan
import importlib.metadata as metadata
import json
from pathlib import Path

for package in ("oreblocks", "numpy", "scipy", "matplotlib"):
    print(package, metadata.version(package))

oreblocks 0.6.3
numpy 2.5.1
scipy 1.18.1
matplotlib 3.11.2


## Data

The notebook reads the committed case manifest. It does not recreate MineLib blocks or infer unavailable block-level schedules.

In [2]:
# 02 bound and feasible plan
manifest_path = Path("../../../data/derived/manifests/newman1-published.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("case:", manifest.get("case_id", manifest.get("case")))
print("scenario:", manifest.get("scenario"))
print("trace bytes:", manifest.get("trace_bytes"))

case: newman1-published
scenario: {'periods': 6, 'discount_rate': 0.08, 'period_one_undiscounted': True, 'n_resources': 2, 'declared': False, 'capacity_fraction': [2.1367, 2.1859], 'limit_per_period': [2000000.0, 1100000.0], 'resource_names': ['mining', 'processing']}
trace bytes: None


## Method

The product helper best_comparable selects only rows eligible for the CPIT envelope. The independent checks below compare the resulting schedule value with the stated bound and calculate the relative gap.

In [3]:
# 02 bound and feasible plan
import sys
sys.path.insert(0, str(Path("../../../data-pipeline").resolve()))
from types import SimpleNamespace
from pipeline.core.manifest import best_comparable
results = [SimpleNamespace(**row) for row in manifest["scoreboard"]]
best = best_comparable(results)
print("comparable result:", best.method, "NPV:", round(best.npv))
assert best.rung in {"classical", "sota", "learned"}

comparable result: sliding-window NPV: 24142200


## Results

This comparison is formulation-specific: PCPSP results and attributed external integer results must remain separately labelled.

In [4]:
# 02 bound and feasible plan
assert manifest_path.is_file()
assert manifest.get("case_id", manifest.get("case")) is not None
print("manifest provenance check: passed")

manifest provenance check: passed


## Summary

Inspected traceable evidence without extrapolating from one case.

## References

- MineLib, benchmark instances and published results: https://minelib.org/v1/Results.xhtml
- Chicoisne et al. (2012), https://doi.org/10.1287/opre.1120.1050
- PhaseFlow case dossier: `docs/use-cases/01_newman1-published.md`